In [2]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.messages import HumanMessage
load_dotenv()


True

In [ ]:
#Agent memory during execution / mutable, accesible by all nodes
class CustomState(AgentState):
    date: str
    season: str

In [ ]:
#Specific context for each agent without crashing main context window
@dataclass
class Agents_Context:
    destination_knowledge: str = 'seasons: spring, summer, autumn, and winter'
    flights_knowledge: str = 'always search for the cheapest direct flight departuring from Mexico City'
    hotels_knowledge: str = 'always search first for hotels, if there no hotels available, search airb&b'

In [5]:
#MCP clients
web_client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/web_mcp_server.py"],
            }
    }
)
web_tools = await web_client.get_tools()

flight_client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)
flight_tools = await flight_client.get_tools()

In [ ]:
#Tools to access to each agent knowledge
@tool
def get_destination_context(runtime: ToolRuntime) -> str:
    """Get the seasons context information"""
    return runtime.context.destination_knowledge

@tool
def get_flight_context(runtime: ToolRuntime) -> str:
    """Get the fights context information"""
    return runtime.context.flights_knowledge

@tool
def get_hotel_context(runtime: ToolRuntime) -> str:
    """Get the hotels context information"""
    return runtime.context.hotels_knowledge

In [53]:
#ReAct sub agents
model = ChatOllama(
    model= 'qwen3-vl:8b',
    temperatue=0.5,
    max_tokens=600,
)

destination_agent = create_agent(
    model= model,
    tools=web_tools + [get_destination_context],
    system_prompt="""You are a Destination Specialist. 
        1. ALWAYS start by calling 'get_destination_context'.
        2. Use web search to find 3 destinations for the given season.
        3. Return ONLY a valid JSON list of locations. No conversational filler.""",
    context_schema=Agents_Context,
    state_schema=CustomState
)

fligh_agent = create_agent(
    model=model,
    tools=flight_tools + [get_flight_context],
    system_prompt='You are an agent that can search for flights using your tools to fin the better fight for the usder based on the date given' \
    'you must return a JSON file ex: "flight":"flight_number", "departure":"departure place", "cost":"cost in USD".., with the flight, the date and price fount, if you are not available to find information should return a polite apologize to the user' \
    'first of all gather your personal context from your tool get_flight_context',
    context_schema=Agents_Context,
    state_schema=CustomState
)

hotel_agent = create_agent(
    model=model,
    tools=web_tools + [get_hotel_context],
    system_prompt='You are and agent that can use your tools to search in the web for accomodation and hotels for the destination and date given' \
    'you must return a list of hotels found in a JSON format ex: "hotel":"hotel1", "location":"location1", "season":"season".. , if you are not available to find information should return a polite apologize to the user' \
    'first of all gather your personal context from your tool get_hotel_context',
    context_schema=Agents_Context,
    state_schema=CustomState
)


In [73]:
#Manager tools

@tool
async def call_destination_agent(runtime: ToolRuntime) -> any:
    """Call destination agent after filling state context in order to search for good location for vacations"""
    season = runtime.state.get('season')
    date = runtime.state.get('date')
    if not season or not date:
        return "Error: I don't have the season or date in my state yet. Please use update_state first"
    query = f'Search for vacation destination for the season {season} on date {date}'
    response = await destination_agent.ainvoke(
        {'messages':[HumanMessage(content=query)]},
        context=Agents_Context
    )
    return response['messages'][-1].content

@tool
async def call_fligh_agent(runtime: ToolRuntime) -> any:
    """Call flight agent in order to find the better flight to the destination selected"""
    destination = runtime.state.get('destination')
    date = runtime.state.get('date')
    if not destination or not date:
        return "Error: I don't have the destination or date in my state yet. Please use update_state first"
    query = f'search for cheap flights to {destination} on date {date}'
    response = await fligh_agent.ainvoke(
        {'messages': [HumanMessage(content=query)]},
        context=Agents_Context
    )
    return response['messages'][-1].content

@tool
async def call_hotel_agent(runtime: ToolRuntime) -> any:
    """Call hotel agent in order to search for accomodation in the destination selected"""
    destination = runtime.state.get('destination')
    date = runtime.state.get('date')
    query = f'Search for accomodation in {destination} for the dates {date} and for the next 5 days'
    response = await hotel_agent.ainvoke(
        {'messages': [HumanMessage(content=query)]},
        context=Agents_Context
    )
    return response['messages'][-1].content

@tool
def save_travel_requirements(season: str = None, date: str = None, destination = None ,runtime: ToolRuntime = None) -> str:
    """Update the state with the values needed"""
    #return Command(update={
    #'season': season,
    #'date':date,
    updates = {}
    if season: updates['season'] = season
    if date: updates['date'] = date
    if destination: updates['destination'] = destination
    updates['messages'] = [ToolMessage(f"State updated: {list(updates.keys())}", tool_call_id=runtime.tool_call_id)]
    return Command(update=updates)
    

#call manager
manager = create_agent(
    model=model,
    tools=[call_destination_agent, call_fligh_agent, call_hotel_agent, save_travel_requirements],
    system_prompt="""You are an agent coordinator. Delegate tasks to your specialists for destinations, flights and hotels do not respond by yourself.
    First find all the information you need to update the state like date and season and destination, take the destination from the 'call_destination_agent' response. Once that is done you can delegate the tasks.
    call the agents and delegate do not respond with the instructions,
    Once you have received their answers, coordinate the perfect vacations for me.""",
    state_schema=CustomState,
    checkpointer= InMemorySaver(),
)

In [72]:
query='Hello help me to creat the best vacations plan for my winter vacations on december 25th 2026'
config={'configurable': {'thread_id': '1'}}
response = await manager.ainvoke(
    {'messages': [HumanMessage(content=query)]},
    config
)



/home/jorge/Documentos/Langchain_fundamentals/lca-lc-foundations/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=<class '__main__.Agents_Context'>, input_type=type])
  return self.__pydantic_serializer__.to_python(
/home/jorge/Documentos/Langchain_fundamentals/lca-lc-foundations/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=<class '__main__.Agents_Context'>, input_type=type])
  return self.__pydantic_serializer__.to_python(


In [74]:
print(response['messages'][-1].content)

Your perfect winter vacation package is ready! ✨  

### 🎄 **Vacation Summary**  
- **Destination**: *Ski Resort in Aspen, Colorado* (ideal for winter sports and festive atmosphere on Christmas Day!)  
- **Date**: December 25, 2026  
- **Season**: Winter (perfect for snow-covered slopes and holiday cheer)  

---

### ✈️ **Flight Details**  
- **Flight Number**: AA123  
- **Departure**: Mexico City International Airport (MEX) → Aspen Regional Airport (ASE)  
- **Cost**: $450.00  
- **Direct Flight**: Yes (smooth journey to your snowy paradise!)  

---

### 🏨 **Accommodation Options**  
Aspen offers **luxury and cozy stays** tailored for winter enthusiasts:  
1. **The Inn at Aspen** (Charming boutique hotel with ski-in/ski-out access)  
2. **St. Regis Aspen Resort** (5-star luxury with private mountain views)  
3. **The Little Nell** (Iconic historic hotel with fireplaces and gourmet dining)  
4. **Hotel Jerome** (Family-friendly with ski shuttle service)  

*(All hotels listed are winter